---
title: Realized variance
execute:
  enabled: true
---

`q.indicator.realized_variance` sums squared returns in one finite return window. The function returns a scalar; callers choose the sampling frequency and rolling or session boundary.

Here it is applied to trailing 21-session daily log-return windows for AAPL and SPY over five years. For production intraday realized variance, pass intraday returns grouped by the required market session.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
returns = pd.DataFrame({symbol: pd.Series(q.indicator.log_returns(prices[symbol]), index=prices.index[1:]) for symbol in prices})
returns.tail()

,AAPL,SPY
datetime,,
2026-07-13,0.006291,-0.007686
2026-07-14,-0.007751,0.003544
2026-07-15,0.039360,0.003956
2026-07-16,0.017435,-0.005433
2026-07-17,0.001439,-0.009946


## Calculate the indicator

A single call reduces one return window to one realized-variance value.

In [2]:
sample = returns["AAPL"].iloc[-21:]
q.indicator.realized_variance(sample)

0.010846543419883226

## Compare with SPY

Applying the scalar estimator to each trailing window creates a time series. Values are multiplied by 10,000 to express squared decimal returns on a percentage scale.

In [3]:
window = 21
comparison = pd.DataFrame(
    {
        symbol: returns[symbol].rolling(window).apply(q.indicator.realized_variance, raw=True)
        for symbol in returns
    }
).mul(10_000)
comparison.columns = [f"{symbol} realized variance" for symbol in comparison]
fig = q.plot.col(
    comparison,
    title=f"AAPL and SPY trailing {window}-session realized variance",
    ylabel="Realized variance (×10,000)",
)
fig.show(renderer="notebook_connected")